# Phase 2 — Trial Acquisition: Category Trial & Sub-brand Trial

**Purpose:** Quantify trial shopper acquisition by size for アリエールジェル vs アタック抗菌EX.

**Key Definitions:**
- **Category Trial** = Shopper with no purchase in the 洗濯洗剤 sub-category in the **prior 12 months** (365 days)
- **Sub-brand Trial** = Shopper with no purchase of a specific sub-brand (e.g., ｱﾘｴｰﾙｼﾞｪﾙ) in the **prior 12 months** (365 days)
- **Repeat** = subsequent same-sub-brand purchase within **180 days** of trial
- **Lapse** = no same-sub-brand purchase within **180 days** of trial

| Step | Description |
|------|-------------|
| 2-1 | Count Category Trial & Sub-brand Trial per size per month |
| 2-2 | Compare trial volumes: アリエールジェル vs アタック抗菌EX (per size) |
| 2-3 | ASP elasticity: monthly ASP vs. sub-brand trial count per size |

**ASP = `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)`**  
**Created:** 2026-02-19

---
## 0. Imports & Connection

In [18]:
import os
import pandas as pd
import numpy as np
import warnings
from dotenv import load_dotenv
import databricks.sql as sql
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import matplotlib.pyplot as plt
import matplotlib.font_manager as fm

warnings.filterwarnings('ignore')
pd.set_option('display.max_rows', 200)
pd.set_option('display.float_format', lambda x: f'{x:,.1f}')

def _find_japanese_font():
    for name in ['MS Gothic', 'MS PGothic', 'Yu Gothic', 'Meiryo', 'IPAexGothic']:
        if name in {f.name for f in fm.fontManager.ttflist}:
            return name
    return None

_jp_font = _find_japanese_font()
if _jp_font:
    plt.rcParams['font.family'] = _jp_font
    print(f'✅ Japanese font: {_jp_font}')

load_dotenv(dotenv_path='../../.env')
DATABRICKS_HOST      = os.getenv('DATABRICKS_HOST')
DATABRICKS_TOKEN     = os.getenv('DATABRICKS_TOKEN')
DATABRICKS_HTTP_PATH = os.getenv('DATABRICKS_HTTP_PATH')
assert all([DATABRICKS_HOST, DATABRICKS_TOKEN, DATABRICKS_HTTP_PATH]), 'Missing .env credentials'
print('✅ Credentials loaded')

def execute_query(query: str) -> pd.DataFrame:
    with sql.connect(server_hostname=DATABRICKS_HOST, http_path=DATABRICKS_HTTP_PATH,
                     access_token=DATABRICKS_TOKEN) as conn:
        with conn.cursor() as cur:
            cur.execute(query)
            result = cur.fetchall()
            columns = [d[0] for d in cur.description]
            return pd.DataFrame(result, columns=columns)

✅ Japanese font: MS Gothic
✅ Credentials loaded


---
## 1. Parameters

In [19]:
ARIEL_GEL   = 'ｱﾘｴｰﾙｼﾞｪﾙ'
ATTACK_EX   = 'ｱﾀｯｸ抗菌EX'
SUB_CAT     = '洗濯洗剤'
CATEGORY    = 'Laundry'

LOOKBACK_START = '2024-01-01'
ANALYSIS_START = '2025-01-01'
ANALYSIS_END   = '2026-01-31'
RENEWAL_MONTH  = '2025-05-01'

RETAILER_CODES = [
    'cds_8005', 'cds_8006', 'cds_8007', 'cds_8008', 'cds_8009',
    'cds_8010', 'cds_8011', 'cds_8012', 'cds_8013',
]
RETAILER_IN = ', '.join(f"'{c}'" for c in RETAILER_CODES)

# ── Size order (physical size: small → large) and exclusions ──────────
SIZE_ORDER     = ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']
EXCLUDED_SIZES = ['ｿﾉﾀ', '詰替通常', '詰替超ｼﾞｬﾝﾎﾞ']

def order_and_filter_sizes(sizes_list):
    """Return sizes in SIZE_ORDER, excluding EXCLUDED_SIZES."""
    sizes_set = set(sizes_list) - set(EXCLUDED_SIZES)
    return [s for s in SIZE_ORDER if s in sizes_set]

print(f'📋 Lookback window: {LOOKBACK_START}')
print(f'📋 Analysis window: {ANALYSIS_START} → {ANALYSIS_END}')
print(f'📋 Renewal breakpoint: {RENEWAL_MONTH}')
print(f'📋 Size order: {SIZE_ORDER}')


📋 Lookback window: 2024-01-01
📋 Analysis window: 2025-01-01 → 2026-01-31
📋 Renewal breakpoint: 2025-05-01
📋 Size order: ['本体通常', '詰替超特大', '詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ', '詰替ﾒｶﾞｼﾞｬﾝﾎﾞ']


---
### 📏 Canonical Definitions (Cross-Notebook Standard)

| Term | Definition | Window |
|------|-----------|--------|
| **Trial Shopper** | A shopper who purchases the sub-brand/category with **no purchase history of that sub-brand/category in the prior 12 months** (365 days). This is a rolling lookback per purchase event, NOT first-ever. | 365-day lookback |
| **Repeat Shopper** | A trial shopper who makes at least one subsequent purchase of the **same sub-brand** within **180 days** (6 months) after their trial event. | 180-day forward window |
| **Lapsed Shopper** | A trial shopper who makes **no subsequent purchase** of the same sub-brand within **180 days** (6 months) after their trial event. | 180-day forward window |
| **ASP** | `SUM(pos_sales_amt) / SUM(pos_unit_sales_qty)` — weighted average selling price. | Per transaction/aggregation |
| **ASP Band (50 JPY bin)** | `FLOOR(ASP / 50) * 50` — ASP floored to nearest 50 JPY. Label: `ASP (50 JPY bin)`. | — |

**Repeat + Lapse are mutually exclusive and exhaustive** within the trial cohort.

> ⚠️ These definitions are enforced consistently across notebooks 00–05 in this analysis.

---
## 2. Step 2-1: Category Trial & Sub-brand Trial Identification

**Logic (12-month lookback — NOT first-ever):**
1. For each shopper purchase in the analysis window, check if they had **any purchase** of the same sub-brand/category in the **preceding 365 days**
2. If no prior purchase exists → that event is a **Trial** event
3. Among multiple qualifying events, the **earliest** in the analysis window is used as the trial date

In [20]:
# ── Sub-brand Trial Shoppers per month per size ───────────────────────
# DEFINITION: A shopper is a sub-brand trial if they purchase the sub-brand
# in the analysis window AND had NO purchase of that sub-brand in the
# preceding 365 days (12-month lookback). This is NOT first-ever logic.
# Uses LAG() window function for efficient gap detection.

subbrand_trial_query = f"""
WITH all_subbrand_purchases AS (
    -- All purchases (lookback + analysis window) for gap calculation
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name            AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3, 4
),
-- LAG to find each purchase's previous purchase date (same shopper + sub-brand)
with_prev AS (
    SELECT *,
        LAG(purchase_date) OVER (
            PARTITION BY shopper_key, sub_brand
            ORDER BY purchase_date
        ) AS prev_purchase_date
    FROM all_subbrand_purchases
),
-- Trial candidate: purchase in analysis window with >365 day gap (or no prior purchase)
trial_candidates AS (
    SELECT *
    FROM with_prev
    WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND (prev_purchase_date IS NULL
           OR DATEDIFF(purchase_date, prev_purchase_date) > 365)
),
-- First trial event per shopper × sub-brand in analysis window
trial_events AS (
    SELECT
        shopper_key,
        sub_brand,
        MIN(purchase_date) AS trial_date
    FROM trial_candidates
    GROUP BY 1, 2
),
-- Join back to get trial size
trial_with_size AS (
    SELECT
        te.shopper_key,
        te.sub_brand,
        te.trial_date,
        tc.size_code
    FROM trial_events te
    INNER JOIN trial_candidates tc
           ON te.shopper_key = tc.shopper_key
          AND te.sub_brand   = tc.sub_brand
          AND te.trial_date  = tc.purchase_date
)
-- Count sub-brand trial shoppers per month per size
SELECT
    DATE_TRUNC('month', ts.trial_date) AS trial_month,
    ts.sub_brand,
    ts.size_code,
    COUNT(DISTINCT ts.shopper_key) AS subbrand_trial_shoppers,
    COUNT(DISTINCT ts.shopper_key) AS size_first_shoppers
FROM trial_with_size ts
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Identifying sub-brand trial shoppers by size (12-month lookback)...', flush=True)
df_subbrand_trial = execute_query(subbrand_trial_query)
df_subbrand_trial['trial_month'] = pd.to_datetime(df_subbrand_trial['trial_month'])
for col in ['subbrand_trial_shoppers', 'size_first_shoppers']:
    df_subbrand_trial[col] = pd.to_numeric(df_subbrand_trial[col])

print(f'✅ {len(df_subbrand_trial)} rows fetched')
print(f'   Ariel trial rows: {len(df_subbrand_trial[df_subbrand_trial["sub_brand"]==ARIEL_GEL])}')
print(f'   Attack trial rows: {len(df_subbrand_trial[df_subbrand_trial["sub_brand"]==ATTACK_EX])}')
print(f'   ℹ️  Trial = no purchase of same sub-brand in prior 365 days (12-month lookback)')

⏳ Identifying sub-brand trial shoppers by size (12-month lookback)...
✅ 179 rows fetched
   Ariel trial rows: 98
   Attack trial rows: 81
   ℹ️  Trial = no purchase of same sub-brand in prior 365 days (12-month lookback)


In [5]:
# ── Category Trial Shoppers ───────────────────────────────────────────
# DEFINITION: Shopper with no purchase in 洗濯洗剤 in the prior 365 days
# (12-month lookback, NOT first-ever category purchase).

cat_trial_query = f"""
WITH all_category_purchases AS (
    SELECT
        idpos.shopper_key,
        prod.jp_sub_brand_alter_lang_name AS sub_brand,
        prod.jp_segment_4_name            AS size_code,
        CAST(idpos.sales_period_group_end_date_part AS DATE) AS purchase_date
    FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
    LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
           ON idpos.prod_key = prod.prod_key
    LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
           ON idpos.shopper_key = shopper.shopper_key
    WHERE idpos.sales_period_group_end_date_part BETWEEN '{LOOKBACK_START}' AND '{ANALYSIS_END}'
      AND idpos.data_provider_code_part IN ({RETAILER_IN})
      AND prod.jp_category_name = '{CATEGORY}'
      AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
      AND shopper.member_ind = 'Y'
      AND idpos.pos_unit_sales_qty > 0
    GROUP BY 1, 2, 3, 4
),
-- LAG to find previous category purchase date (any sub-brand in category)
with_prev_cat AS (
    SELECT *,
        LAG(purchase_date) OVER (
            PARTITION BY shopper_key
            ORDER BY purchase_date
        ) AS prev_cat_date
    FROM all_category_purchases
),
-- Category trial: purchase in analysis window with >365 day gap (or no prior)
cat_trial_candidates AS (
    SELECT *
    FROM with_prev_cat
    WHERE purchase_date BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
      AND (prev_cat_date IS NULL
           OR DATEDIFF(purchase_date, prev_cat_date) > 365)
      AND sub_brand IN ('{ARIEL_GEL}', '{ATTACK_EX}')
),
-- First category trial per shopper in analysis window
cat_trial_events AS (
    SELECT
        shopper_key,
        MIN(purchase_date) AS cat_trial_date
    FROM cat_trial_candidates
    GROUP BY 1
),
-- Get detail (sub-brand, size) from the trial date
cat_trial_detail AS (
    SELECT
        cte.shopper_key,
        ctc.sub_brand,
        ctc.size_code,
        cte.cat_trial_date AS purchase_date
    FROM cat_trial_events cte
    INNER JOIN cat_trial_candidates ctc
           ON cte.shopper_key    = ctc.shopper_key
          AND cte.cat_trial_date = ctc.purchase_date
)
SELECT
    DATE_TRUNC('month', ctd.purchase_date) AS trial_month,
    ctd.sub_brand,
    ctd.size_code,
    COUNT(DISTINCT ctd.shopper_key) AS category_trial_shoppers
FROM cat_trial_detail ctd
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Identifying category trial shoppers (12-month lookback)...', flush=True)
df_cat_trial = execute_query(cat_trial_query)
df_cat_trial['trial_month'] = pd.to_datetime(df_cat_trial['trial_month'])
df_cat_trial['category_trial_shoppers'] = pd.to_numeric(df_cat_trial['category_trial_shoppers'])

print(f'✅ {len(df_cat_trial)} rows')
print(f'   Ariel category trial: {df_cat_trial[df_cat_trial["sub_brand"]==ARIEL_GEL]["category_trial_shoppers"].sum():,.0f}')
print(f'   Attack category trial: {df_cat_trial[df_cat_trial["sub_brand"]==ATTACK_EX]["category_trial_shoppers"].sum():,.0f}')
print(f'   ℹ️  Category trial = no category purchase in prior 365 days (12-month lookback)')

⏳ Identifying category trial shoppers (12-month lookback)...
✅ 179 rows
   Ariel category trial: 1,304,178
   Attack category trial: 2,499,922
   ℹ️  Category trial = no category purchase in prior 365 days (12-month lookback)


In [21]:
# ── Merge both trial types ────────────────────────────────────────────
df_trial = df_subbrand_trial.merge(
    df_cat_trial,
    on=['trial_month', 'sub_brand', 'size_code'],
    how='outer'
).fillna(0)

for col in ['subbrand_trial_shoppers', 'size_first_shoppers', 'category_trial_shoppers']:
    df_trial[col] = df_trial[col].astype(int)

print('Combined Trial Summary (monthly):')
print('=' * 80)
summary = df_trial.groupby(['sub_brand', 'size_code']).agg(
    total_cat_trial=('category_trial_shoppers', 'sum'),
    total_subbrand_trial=('subbrand_trial_shoppers', 'sum'),
    total_size_first=('size_first_shoppers', 'sum'),
).reset_index()

print('\n▶ アリエールジェル')
print(summary[summary['sub_brand'] == ARIEL_GEL].to_string(index=False))
print('\n▶ アタック抗菌EX')
print(summary[summary['sub_brand'] == ATTACK_EX].to_string(index=False))

Combined Trial Summary (monthly):

▶ アリエールジェル
sub_brand     size_code  total_cat_trial  total_subbrand_trial  total_size_first
ｱﾘｴｰﾙｼﾞｪﾙ          本体通常           219389                563730            563730
ｱﾘｴｰﾙｼﾞｪﾙ         詰替超特大           378346                847916            847916
ｱﾘｴｰﾙｼﾞｪﾙ 詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           355302                698764            698764
ｱﾘｴｰﾙｼﾞｪﾙ     詰替超ｼﾞｬﾝﾎﾞ             3152                  5485              5485
ｱﾘｴｰﾙｼﾞｪﾙ          詰替通常               54                   145               145
ｱﾘｴｰﾙｼﾞｪﾙ  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ           200323                404075            404075
ｱﾘｴｰﾙｼﾞｪﾙ   詰替ﾒｶﾞｼﾞｬﾝﾎﾞ           131715                249969            249969
ｱﾘｴｰﾙｼﾞｪﾙ           ｿﾉﾀ            15897                 31863             31863

▶ アタック抗菌EX
sub_brand     size_code  total_cat_trial  total_subbrand_trial  total_size_first
 ｱﾀｯｸ抗菌EX          本体通常           199483                486513            486513
 ｱﾀｯｸ抗菌EX         詰替超特大           574417           

In [22]:

# ── [DE] Monthly total unique buyers per brand × size ──────────────────
# Required as denominator for Trial Rate = trial shoppers / total shoppers
# Uses ANALYSIS_START→END window; deduplicated at monthly level in SQL
total_buyers_query = f"""
SELECT
    DATE_TRUNC('month', CAST(idpos.sales_period_group_end_date_part AS DATE)) AS trial_month,
    prod.jp_sub_brand_alter_lang_name AS sub_brand,
    prod.jp_segment_4_name            AS size_code,
    COUNT(DISTINCT idpos.shopper_key) AS total_shoppers
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3
ORDER BY 1, 2, 3
"""

print('⏳ Fetching monthly total unique buyers per brand × size...', flush=True)
df_monthly_buyers = execute_query(total_buyers_query)
df_monthly_buyers['trial_month'] = pd.to_datetime(df_monthly_buyers['trial_month'])
df_monthly_buyers['total_shoppers'] = pd.to_numeric(df_monthly_buyers['total_shoppers'])

# Merge trial rate denominator into df_trial
df_trial = df_trial.merge(df_monthly_buyers, on=['trial_month', 'sub_brand', 'size_code'], how='left')
df_trial['trial_rate'] = (
    df_trial['subbrand_trial_shoppers'] / df_trial['total_shoppers'].replace(0, np.nan) * 100
).round(2)

print(f'✅ {len(df_monthly_buyers):,} monthly buyer rows fetched')
print(f'   Trial rate range (Ariel): '
      f'{df_trial[df_trial["sub_brand"]==ARIEL_GEL]["trial_rate"].min():.1f}% – '
      f'{df_trial[df_trial["sub_brand"]==ARIEL_GEL]["trial_rate"].max():.1f}%')
print(f'   Trial rate range (Attack): '
      f'{df_trial[df_trial["sub_brand"]==ATTACK_EX]["trial_rate"].min():.1f}% – '
      f'{df_trial[df_trial["sub_brand"]==ATTACK_EX]["trial_rate"].max():.1f}%')


⏳ Fetching monthly total unique buyers per brand × size...
✅ 182 monthly buyer rows fetched
   Trial rate range (Ariel): 15.2% – 72.2%
   Trial rate range (Attack): 16.1% – 100.0%


---
## 3. Step 2-2: Trial Volume Comparison by Size

In [23]:
# ── Sub-brand trial: monthly trend per size, Ariel vs Attack ──────────
for brand_name, brand_code in [('アリエールジェル', ARIEL_GEL), ('アタック抗菌EX', ATTACK_EX)]:
    brand_data = df_trial[
        (df_trial['sub_brand'] == brand_code) &
        (~df_trial['size_code'].isin(EXCLUDED_SIZES))
    ].copy()
    sizes = order_and_filter_sizes(brand_data['size_code'].unique())

    if len(sizes) == 0:
        print(f'⚠️ No data for {brand_name}')
        continue

    n_s = len(sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c

    fig = make_subplots(rows=n_r, cols=n_c,
                        subplot_titles=[f'{brand_name} {s}' for s in sizes],
                        vertical_spacing=0.1, horizontal_spacing=0.08)

    for idx, size in enumerate(sizes):
        row = idx // n_c + 1
        col = idx % n_c + 1
        subset = brand_data[brand_data['size_code'] == size].sort_values('trial_month')

        fig.add_trace(go.Bar(x=subset['trial_month'], y=subset['subbrand_trial_shoppers'],
                             name=f'{size} Sub-brand Trial', marker_color='#FF6B6B',
                             showlegend=(idx==0)),
                      row=row, col=col)
        fig.add_trace(go.Bar(x=subset['trial_month'], y=subset['category_trial_shoppers'],
                             name=f'{size} Category Trial', marker_color='#4ECDC4',
                             showlegend=(idx==0)),
                      row=row, col=col)

    fig.update_layout(height=300*n_r, barmode='group',
                      title_text=f'{brand_name}: Monthly Trial Shoppers by Size (Category vs Sub-brand)',
                      template='plotly_white')
    fig.update_yaxes(title_text='Trial Shoppers')
    fig.show()


In [24]:

# ── [DS] Cell 12: Combined monthly trial — Ariel vs Attack per Size ───────────────
# Bars  = Sub-brand trial shoppers (left Y)
# Lines = Trial Rate % (left Y dotted)  |  Index Ariel/Attack×100 (right Y dashed green)
# Size order and exclusions from HANDOVER parameters

BRAND_COLOR = {ARIEL_GEL: '#1E90FF', ATTACK_EX: '#FF6347'}
BRAND_LABEL = {ARIEL_GEL: 'Ariel Gel', ATTACK_EX: 'Attack 抗菌EX'}

combined_trial = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].copy()
comb_sizes = order_and_filter_sizes(combined_trial['size_code'].unique())

if len(comb_sizes) == 0:
    print('⚠️ No sizes available for combined trial chart')
else:
    # ── Pre-compute per-month per-size index (Ariel trial / Attack trial × 100) ──
    _piv = combined_trial.groupby(['trial_month', 'sub_brand', 'size_code']).agg(
        subbrand_trial=('subbrand_trial_shoppers', 'sum'),
        trial_rate=('trial_rate', 'mean')
    ).reset_index()
    _a = _piv[_piv['sub_brand'] == ARIEL_GEL][['trial_month', 'size_code', 'subbrand_trial', 'trial_rate']].rename(
        columns={'subbrand_trial': 'ariel_trial', 'trial_rate': 'ariel_rate'})
    _b = _piv[_piv['sub_brand'] == ATTACK_EX][['trial_month', 'size_code', 'subbrand_trial']].rename(
        columns={'subbrand_trial': 'attack_trial'})
    _idx_df = _a.merge(_b, on=['trial_month', 'size_code'], how='outer').sort_values('trial_month')
    _idx_df['trial_index'] = (_idx_df['ariel_trial'] / _idx_df['attack_trial'].replace(0, np.nan) * 100).round(1)

    n_s = len(comb_sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c

    # secondary_y=True enables dual-axis per subplot
    specs = [[{'secondary_y': True}] * n_c for _ in range(n_r)]
    fig_comb = make_subplots(
        rows=n_r, cols=n_c,
        specs=specs,
        subplot_titles=comb_sizes,
        vertical_spacing=0.15, horizontal_spacing=0.14
    )

    for idx, size in enumerate(comb_sizes):
        r = idx // n_c + 1
        c = idx % n_c + 1

        # ── Bars: trial shoppers per brand ────────────────────────────
        for brand in [ARIEL_GEL, ATTACK_EX]:
            sub = combined_trial[
                (combined_trial['sub_brand'] == brand) &
                (combined_trial['size_code'] == size)
            ].sort_values('trial_month')
            if len(sub) == 0:
                continue
            fig_comb.add_trace(go.Bar(
                x=sub['trial_month'], y=sub['subbrand_trial_shoppers'],
                name=BRAND_LABEL[brand],
                marker_color=BRAND_COLOR[brand],
                opacity=0.85,
                showlegend=(idx == 0),
                legendgroup=BRAND_LABEL[brand]
            ), row=r, col=c, secondary_y=False)

            # ── Dotted line: trial rate % on left Y ───────────────────
            if 'trial_rate' in sub.columns and sub['trial_rate'].notna().any():
                fig_comb.add_trace(go.Scatter(
                    x=sub['trial_month'], y=sub['trial_rate'],
                    name=f'{BRAND_LABEL[brand]} Trial Rate%',
                    mode='lines+markers',
                    line=dict(color=BRAND_COLOR[brand], dash='dot', width=1.5),
                    marker=dict(size=4),
                    showlegend=(idx == 0),
                    legendgroup=f'{BRAND_LABEL[brand]} Rate'
                ), row=r, col=c, secondary_y=False)

        # ── Right Y: Ariel/Attack Index ────────────────────────────────
        _s = _idx_df[_idx_df['size_code'] == size].copy()
        if len(_s) > 0 and _s['trial_index'].notna().any():
            fig_comb.add_trace(go.Scatter(
                x=_s['trial_month'], y=_s['trial_index'],
                name='Index Ariel/Attack×100',
                mode='lines+markers',
                line=dict(color='#2CA02C', dash='dash', width=2),
                marker=dict(size=6, symbol='diamond'),
                showlegend=(idx == 0),
                legendgroup='Index'
            ), row=r, col=c, secondary_y=True)
            # Reference line at 100 (parity)
            fig_comb.add_hline(y=100, line_dash='dot', line_color='gray',
                               line_width=1, row=r, col=c, secondary_y=True)

        fig_comb.update_yaxes(title_text='Trial Shoppers / Trial Rate%',
                               row=r, col=c, secondary_y=False)
        fig_comb.update_yaxes(title_text='Index (Ariel/Attack×100)',
                               row=r, col=c, secondary_y=True,
                               showgrid=False)

    fig_comb.update_layout(
        height=420 * n_r,
        barmode='group',
        title_text=(
            'Monthly Sub-brand Trial: アリエールジェル vs アタック抗菌EX (per Size)<br>'
            '<sup>Bars=Trial Shoppers | Dotted=Trial Rate% | Green dash=Index Ariel÷Attack×100 | Gray dot=parity 100</sup>'
        ),
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=-0.15, x=0)
    )
    fig_comb.show()
    print('\n📊 Summary (full period, incl. Index & Trial Rate):')
    for size in comb_sizes:
        a = combined_trial[(combined_trial['sub_brand'] == ARIEL_GEL) & (combined_trial['size_code'] == size)]
        b = combined_trial[(combined_trial['sub_brand'] == ATTACK_EX) & (combined_trial['size_code'] == size)]
        a_t = a['subbrand_trial_shoppers'].sum()
        b_t = b['subbrand_trial_shoppers'].sum()
        a_r = a['trial_rate'].mean() if 'trial_rate' in a.columns else np.nan
        b_r = b['trial_rate'].mean() if 'trial_rate' in b.columns else np.nan
        print(f'  {size:30s}  Ariel {a_t:>7,} ({a_r:.1f}%)  Attack {b_t:>7,} ({b_r:.1f}%)  '
              f'Index {a_t/b_t*100:.0f}' if b_t > 0 else f'  {size:30s}  Ariel {a_t:>7,}  Attack 0')



📊 Summary (full period, incl. Index & Trial Rate):
  本体通常                            Ariel 563,730 (40.8%)  Attack 486,513 (44.2%)  Index 116
  詰替超特大                           Ariel 847,916 (26.4%)  Attack 980,058 (22.0%)  Index 87
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                    Ariel 404,075 (22.2%)  Attack 745,737 (19.3%)  Index 54
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   Ariel 698,764 (20.6%)  Attack 1,543,082 (20.1%)  Index 45
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                     Ariel 249,969 (22.2%)  Attack 502,211 (26.1%)  Index 50


In [25]:

# ── [DS] Cell 13: Head-to-head sub-brand trial by size ────────────────

h2h = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].groupby(
    ['sub_brand', 'size_code']
).agg(
    subbrand_trial=('subbrand_trial_shoppers', 'sum'),
    avg_trial_rate=('trial_rate', 'mean')
).reset_index()

h2h_sizes = order_and_filter_sizes(h2h['size_code'].unique())
h2h = h2h[h2h['size_code'].isin(h2h_sizes)].copy()
h2h['size_order'] = h2h['size_code'].map({s: i for i, s in enumerate(h2h_sizes)})
h2h = h2h.sort_values('size_order')

_ha = h2h[h2h['sub_brand'] == ARIEL_GEL][['size_code', 'subbrand_trial', 'avg_trial_rate']].rename(
    columns={'subbrand_trial': 'ariel_t', 'avg_trial_rate': 'ariel_rate'})
_hb = h2h[h2h['sub_brand'] == ATTACK_EX][['size_code', 'subbrand_trial', 'avg_trial_rate']].rename(
    columns={'subbrand_trial': 'attack_t', 'avg_trial_rate': 'attack_rate'})
h2h_idx = _ha.merge(_hb, on='size_code', how='outer')
h2h_idx['size_idx'] = (h2h_idx['ariel_t'] / h2h_idx['attack_t'].replace(0, np.nan) * 100).round(1)
h2h_idx['size_order'] = h2h_idx['size_code'].map({s: i for i, s in enumerate(h2h_sizes)})
h2h_idx = h2h_idx.sort_values('size_order')

sb_ariel_ttl = df_trial[df_trial['sub_brand'] == ARIEL_GEL]['subbrand_trial_shoppers'].sum()
sb_attack_ttl = df_trial[df_trial['sub_brand'] == ATTACK_EX]['subbrand_trial_shoppers'].sum()
sb_benchmark_idx = (sb_ariel_ttl / sb_attack_ttl * 100) if sb_attack_ttl > 0 else np.nan

# ── Natural, attractive palette ────────────────────────────────────────
ARIEL_COL  = '#2980B9'   # sky blue   – calm, brand-adjacent
ATTACK_COL = '#E67E22'   # amber      – warm earth tone, visually distinct
INDEX_COL  = '#27AE60'   # leaf green – growth/comparison signal
BENCH_COL  = '#7F8C8D'   # slate gray – neutral reference

# ── K / M formatter ────────────────────────────────────────────────────
def fmt_km(v):
    if v >= 1_000_000: return f'{v/1_000_000:.1f}M'
    if v >= 1_000:     return f'{v/1_000:.1f}K'
    return str(int(v))

fig_h2h = make_subplots(specs=[[{'secondary_y': True}]])

for brand_code, brand_label, color in [
    (ARIEL_GEL, 'Ariel Gel  ─  Trial Shoppers', ARIEL_COL),
    (ATTACK_EX, 'Attack 抗菌EX  ─  Trial Shoppers', ATTACK_COL)
]:
    d = h2h[h2h['sub_brand'] == brand_code]
    fig_h2h.add_trace(go.Bar(
        x=d['size_code'], y=d['subbrand_trial'],
        name=brand_label,
        marker_color=color, opacity=0.80,
        text=[fmt_km(v) for v in d['subbrand_trial']],
        textposition='outside', textfont=dict(size=12, color=color)
    ), secondary_y=False)
    if 'avg_trial_rate' in d.columns and d['avg_trial_rate'].notna().any():
        trial_label = 'Ariel Gel' if brand_code == ARIEL_GEL else 'Attack 抗菌EX'
        fig_h2h.add_trace(go.Scatter(
            x=d['size_code'], y=d['avg_trial_rate'],
            mode='markers+text',
            name=f'{trial_label}  ○  Trial Rate %',
            marker=dict(color=color, size=13, symbol='circle-open', line=dict(width=2.5)),
            text=[f'{v:.1f}%' for v in d['avg_trial_rate']],
            textposition='top center',
            textfont=dict(size=11, color=color)
        ), secondary_y=True)

# Per-size index line
fig_h2h.add_trace(go.Scatter(
    x=h2h_idx['size_code'], y=h2h_idx['size_idx'],
    mode='lines+markers+text',
    name='◇ Size Index  (Ariel ÷ Attack × 100)',
    line=dict(color=INDEX_COL, dash='dash', width=2.5),
    marker=dict(size=10, symbol='diamond', color=INDEX_COL),
    text=[f'{v:.0f}' for v in h2h_idx['size_idx']],
    textposition='bottom center',
    textfont=dict(color=INDEX_COL, size=11)
), secondary_y=True)

# Benchmark line – add dummy scatter so it appears in legend
if not np.isnan(sb_benchmark_idx):
    fig_h2h.add_hline(
        y=sb_benchmark_idx, line_dash='longdash', line_color=BENCH_COL, line_width=2,
        secondary_y=True
    )
    fig_h2h.add_trace(go.Scatter(
        x=[None], y=[None], mode='lines',
        name=f'── Sub-brand Total Index = {sb_benchmark_idx:.0f}  (benchmark)',
        line=dict(color=BENCH_COL, dash='longdash', width=2),
        showlegend=True
    ), secondary_y=True)

# Parity line
fig_h2h.add_hline(y=100, line_dash='dot', line_color='#BDC3C7', line_width=1.5, secondary_y=True)
fig_h2h.add_trace(go.Scatter(
    x=[None], y=[None], mode='lines',
    name='·· Parity = 100',
    line=dict(color='#BDC3C7', dash='dot', width=1.5),
    showlegend=True
), secondary_y=True)

fig_h2h.update_layout(
    title=dict(
        text='Head-to-Head: Sub-brand Trial Shoppers by Size (Jan 2025 – Jan 2026)',
        font=dict(size=16)
    ),
    barmode='group',
    template='plotly_white',
    height=600,
    font=dict(size=13),
    xaxis=dict(categoryorder='array', categoryarray=h2h_sizes, tickfont=dict(size=13)),
    legend=dict(
        orientation='h', yanchor='bottom', y=-0.30, x=0,
        font=dict(size=12), bgcolor='rgba(255,255,255,0.8)',
        bordercolor='#E0E0E0', borderwidth=1
    )
)
fig_h2h.update_yaxes(title_text='Trial Shoppers', secondary_y=False,
                     title_font=dict(size=13), tickfont=dict(size=12))
fig_h2h.update_yaxes(title_text='Index / Trial Rate %', secondary_y=True, showgrid=False,
                     title_font=dict(size=13), tickfont=dict(size=12))
fig_h2h.show()


---
## 2b. Monthly Trial Comparison: アリエールジェル vs アタック抗菌EX per Size

In [11]:

# ── [DS] Cell 13: Monthly trial comparison — lines per size + 2nd Y (trial rate) ─
# Primary Y  = Sub-brand trial shoppers (solid lines)
# Secondary Y= Trial Rate % (dotted lines)  |  Index Ariel/Attack×100 (dashed green)
# BENCHMARK  = Sub-brand total (all sizes) Ariel÷Attack×100 shown as bold gray dashed line

# 1. Apply SIZE_ORDER + EXCLUDED_SIZES
major_sizes = order_and_filter_sizes(
    [s for s in (set(df_trial[df_trial['sub_brand'] == ARIEL_GEL]['size_code']) &
                  set(df_trial[df_trial['sub_brand'] == ATTACK_EX]['size_code']))
     if df_trial[(df_trial['size_code'] == s)]['subbrand_trial_shoppers'].sum() > 500
     and s not in EXCLUDED_SIZES]
)

# 2. Sub-brand level benchmark: total Ariel / total Attack × 100 per month (all sizes)
_sb = df_trial.groupby(['trial_month', 'sub_brand'])['subbrand_trial_shoppers'].sum().reset_index()
_sb_a = _sb[_sb['sub_brand'] == ARIEL_GEL][['trial_month', 'subbrand_trial_shoppers']].rename(
    columns={'subbrand_trial_shoppers': 'ariel_total'})
_sb_b = _sb[_sb['sub_brand'] == ATTACK_EX][['trial_month', 'subbrand_trial_shoppers']].rename(
    columns={'subbrand_trial_shoppers': 'attack_total'})
sb_bench = _sb_a.merge(_sb_b, on='trial_month', how='inner').sort_values('trial_month')
sb_bench['sb_index'] = (sb_bench['ariel_total'] / sb_bench['attack_total'].replace(0, np.nan) * 100).round(1)

# 3. Per-size index pivot for secondary axis
_piv2 = df_trial[~df_trial['size_code'].isin(EXCLUDED_SIZES)].groupby(
    ['trial_month', 'sub_brand', 'size_code']
).agg(
    subbrand_trial=('subbrand_trial_shoppers', 'sum'),
    trial_rate=('trial_rate', 'mean')
).reset_index()
_a2 = _piv2[_piv2['sub_brand'] == ARIEL_GEL][['trial_month', 'size_code', 'subbrand_trial', 'trial_rate']].rename(
    columns={'subbrand_trial': 'ariel_trial', 'trial_rate': 'ariel_rate'})
_b2 = _piv2[_piv2['sub_brand'] == ATTACK_EX][['trial_month', 'size_code', 'subbrand_trial', 'trial_rate']].rename(
    columns={'subbrand_trial': 'attack_trial', 'trial_rate': 'attack_rate'})
_idx2 = _a2.merge(_b2, on=['trial_month', 'size_code'], how='outer').sort_values('trial_month')
_idx2['trial_index'] = (_idx2['ariel_trial'] / _idx2['attack_trial'].replace(0, np.nan) * 100).round(1)

if len(major_sizes) == 0:
    print('⚠️ No qualifying sizes (both brands, >500 trial shoppers)')
else:
    n_s = len(major_sizes)
    n_c = min(2, n_s)
    n_r = (n_s + n_c - 1) // n_c

    specs = [[{'secondary_y': True}] * n_c for _ in range(n_r)]
    fig = make_subplots(
        rows=n_r, cols=n_c,
        specs=specs,
        subplot_titles=major_sizes,
        vertical_spacing=0.15, horizontal_spacing=0.14
    )

    for idx, size in enumerate(major_sizes):
        row = idx // n_c + 1
        col = idx % n_c + 1

        # ── Primary Y: trial shoppers (solid lines) ───────────────────
        for brand_code, brand_label, color in [
            (ARIEL_GEL, 'Ariel Gel', '#1E90FF'),
            (ATTACK_EX, 'Attack 抗菌EX', '#FF6347')
        ]:
            sub = df_trial[(df_trial['sub_brand'] == brand_code) &
                           (df_trial['size_code'] == size)].sort_values('trial_month')
            if len(sub) == 0:
                continue
            fig.add_trace(go.Scatter(
                x=sub['trial_month'], y=sub['subbrand_trial_shoppers'],
                mode='lines+markers', name=brand_label,
                line=dict(color=color, width=2.5),
                marker=dict(size=6),
                showlegend=(idx == 0),
                legendgroup=brand_label
            ), row=row, col=col, secondary_y=False)

            # ── Secondary Y: trial rate % (dotted) ────────────────────
            if 'trial_rate' in sub.columns and sub['trial_rate'].notna().any():
                fig.add_trace(go.Scatter(
                    x=sub['trial_month'], y=sub['trial_rate'],
                    mode='lines',
                    name=f'{brand_label} Trial Rate%',
                    line=dict(color=color, dash='dot', width=1.5),
                    showlegend=(idx == 0),
                    legendgroup=f'{brand_label} Rate'
                ), row=row, col=col, secondary_y=True)

        # ── Secondary Y: per-size index (Ariel/Attack×100) ────────────
        _si = _idx2[_idx2['size_code'] == size]
        if len(_si) > 0 and _si['trial_index'].notna().any():
            fig.add_trace(go.Scatter(
                x=_si['trial_month'], y=_si['trial_index'],
                mode='lines+markers',
                name='Size Index (Ariel/Attack×100)',
                line=dict(color='#2CA02C', dash='dash', width=2),
                marker=dict(size=5, symbol='diamond'),
                showlegend=(idx == 0),
                legendgroup='Size Index'
            ), row=row, col=col, secondary_y=True)

        # ── BENCHMARK: Sub-brand total index (all sizes, bold gray) ───
        if len(sb_bench) > 0:
            fig.add_trace(go.Scatter(
                x=sb_bench['trial_month'], y=sb_bench['sb_index'],
                mode='lines',
                name='Sub-brand Total Index (benchmark)',
                line=dict(color='#888888', dash='longdash', width=2),
                showlegend=(idx == 0),
                legendgroup='Subbrand Benchmark'
            ), row=row, col=col, secondary_y=True)

        # Parity reference at 100
        fig.add_hline(y=100, line_dash='dot', line_color='lightgray',
                      line_width=1, row=row, col=col, secondary_y=True)

        fig.update_yaxes(title_text='Trial Shoppers',
                         row=row, col=col, secondary_y=False)
        fig.update_yaxes(title_text='Index / Trial Rate%',
                         row=row, col=col, secondary_y=True, showgrid=False)

    fig.update_layout(
        height=400 * n_r,
        title_text=(
            'Monthly Sub-brand Trial: アリエールジェル vs アタック抗菌EX (per Size)<br>'
            '<sup>Solid=Trial Shoppers | Dotted=Trial Rate% | Green dash=Size-Level Index | '
            'Gray dash=Sub-brand Total Index (benchmark)</sup>'
        ),
        template='plotly_white',
        legend=dict(orientation='h', yanchor='bottom', y=-0.18, x=0)
    )
    fig.show()

# ── Summary table ─────────────────────────────────────────────────────
print('\nMonthly Trial Comparison (major sizes):')
_sb_total_a = sb_bench['ariel_total'].sum()
_sb_total_b = sb_bench['attack_total'].sum()
print(f'  {"[Sub-brand Total]":30s} Ariel: {_sb_total_a:>8,}  Attack: {_sb_total_b:>8,}  '
      f'Index: {_sb_total_a/_sb_total_b*100:.0f}' if _sb_total_b > 0 else '---')
print('-' * 80)
for size in major_sizes:
    ariel_total = df_trial[(df_trial['sub_brand'] == ARIEL_GEL) &
                           (df_trial['size_code'] == size)]['subbrand_trial_shoppers'].sum()
    attack_total = df_trial[(df_trial['sub_brand'] == ATTACK_EX) &
                            (df_trial['size_code'] == size)]['subbrand_trial_shoppers'].sum()
    ariel_rate = df_trial[(df_trial['sub_brand'] == ARIEL_GEL) &
                          (df_trial['size_code'] == size)]['trial_rate'].mean() if 'trial_rate' in df_trial.columns else np.nan
    attack_rate = df_trial[(df_trial['sub_brand'] == ATTACK_EX) &
                           (df_trial['size_code'] == size)]['trial_rate'].mean() if 'trial_rate' in df_trial.columns else np.nan
    idx_val = ariel_total / attack_total * 100 if attack_total > 0 else 0
    print(f'  {size:30s} Ariel: {ariel_total:>8,} ({ariel_rate:.1f}%)  '
          f'Attack: {attack_total:>8,} ({attack_rate:.1f}%)  Index: {idx_val:.0f}')



Monthly Trial Comparison (major sizes):
  [Sub-brand Total]              Ariel: 2,801,947  Attack: 4,265,634  Index: 66
--------------------------------------------------------------------------------
  本体通常                           Ariel:  563,722 (40.8%)  Attack:  486,464 (44.2%)  Index: 116
  詰替超特大                          Ariel:  847,925 (26.4%)  Attack:  980,061 (22.0%)  Index: 87
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   Ariel:  404,075 (22.2%)  Attack:  745,742 (19.3%)  Index: 54
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                  Ariel:  698,763 (20.6%)  Attack: 1,543,123 (20.1%)  Index: 45
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                    Ariel:  249,969 (22.2%)  Attack:  502,211 (26.1%)  Index: 50


---
## 4. Step 2-3: ASP Elasticity — Does Lower ASP Drive More Trials?

In [26]:
# ── Fetch WEEKLY RETAILER-LEVEL ASP + units per size ──────────────────
# Weekly × retailer granularity is the standard approach for price elasticity
# — monthly national averages smooth out too much variation
asp_weekly_query = f"""
SELECT
    CAST(idpos.sales_period_group_end_date_part AS DATE) AS week_end,
    idpos.data_provider_code_part AS retailer,
    prod.jp_sub_brand_alter_lang_name   AS sub_brand,
    prod.jp_segment_4_name              AS size_code,
    SUM(idpos.pos_sales_amt) / SUM(idpos.pos_unit_sales_qty) AS weighted_asp,
    SUM(idpos.pos_unit_sales_qty) AS total_units,
    COUNT(DISTINCT idpos.shopper_key) AS buyer_count
FROM cdl_customer_prod.gold_customer_loyalty.loyalty_transact_fct_v1_vw idpos
LEFT JOIN id_pos_ai_1.prod_dim_ext_vw prod
       ON idpos.prod_key = prod.prod_key
LEFT JOIN id_pos_ai_1.shopper_dim_generic_vw shopper
       ON idpos.shopper_key = shopper.shopper_key
WHERE idpos.sales_period_group_end_date_part BETWEEN '{ANALYSIS_START}' AND '{ANALYSIS_END}'
  AND idpos.data_provider_code_part IN ({RETAILER_IN})
  AND prod.jp_category_name = '{CATEGORY}'
  AND prod.jp_sub_category_alter_lang_name = '{SUB_CAT}'
  AND prod.jp_sub_brand_alter_lang_name IN ('{ARIEL_GEL}', '{ATTACK_EX}')
  AND shopper.member_ind = 'Y'
  AND idpos.pos_unit_sales_qty > 0
GROUP BY 1, 2, 3, 4
ORDER BY 1, 2, 3, 4
"""

print('⏳ Fetching weekly retailer-level ASP + units (more granular than monthly)...', flush=True)
df_asp_weekly = execute_query(asp_weekly_query)
df_asp_weekly['week_end'] = pd.to_datetime(df_asp_weekly['week_end'])
for c in ['weighted_asp', 'total_units', 'buyer_count']:
    df_asp_weekly[c] = pd.to_numeric(df_asp_weekly[c])
print(f'✅ {len(df_asp_weekly):,} rows (weekly × retailer × brand × size)')
print(f'   Weeks: {df_asp_weekly["week_end"].nunique()}, Retailers: {df_asp_weekly["retailer"].nunique()}')

# Also keep monthly ASP for backward compat (aggregate from weekly)
df_asp = df_asp_weekly.copy()
df_asp['month'] = df_asp['week_end'].dt.to_period('M').dt.to_timestamp()
df_asp = df_asp.groupby(['month', 'sub_brand', 'size_code']).agg(
    weighted_asp=('weighted_asp', 'mean'),
    total_units=('total_units', 'sum')
).reset_index()

⏳ Fetching weekly retailer-level ASP + units (more granular than monthly)...
✅ 29,713 rows (weekly × retailer × brand × size)
   Weeks: 396, Retailers: 9


In [27]:

# ── [DS] Cell 18: Price Point Productivity Analysis ───────────────────
# Productivity = Trial Rate % per 50-JPY ASP bin (buyers ÷ category shoppers)
# Uses weekly ASP data joined to monthly trial-rate so the FULL price range
# is retained.  Monthly-only merge collapsed ~52 weekly obs → ~13, leaving
# many price bins below the threshold and cutting the edges of the range.
#
# Productivity Index = bin_trial_rate / overall_avg_trial_rate × 100
#   > 100 → above-average conversion at that price
#   < 100 → below-average conversion
#
# Chart A: Productivity Index bars per size (full price range)
# Chart B: Weekly ASP vs Buyers scatter (raw execution view)

# ── Setup: monthly ASP+trial merge ────────────────────────────────────
df_asp_monthly_agg = df_asp_weekly.groupby(
    [df_asp_weekly['week_end'].dt.to_period('M').dt.to_timestamp().rename('trial_month'),
     'sub_brand', 'size_code']
).agg(
    weighted_asp=('weighted_asp', 'mean'),
    total_units=('total_units', 'sum')
).reset_index()

df_trial_compat = df_trial.copy()
if hasattr(df_trial_compat['trial_month'].dt, 'tz') and df_trial_compat['trial_month'].dt.tz is not None:
    df_trial_compat['trial_month'] = df_trial_compat['trial_month'].dt.tz_localize(None)

df_elasticity = df_trial_compat.merge(
    df_asp_monthly_agg, on=['trial_month', 'sub_brand', 'size_code'], how='inner'
)

# ── Weekly data enriched with trial_rate (full price range for Chart A) ──
# Join weekly ASP rows → monthly trial_rate by month key.
# This multiplies ~52 weekly obs per size×month, giving far more obs per
# 50-JPY bin than the monthly-only path and preserving the full price range.
ariel_weekly_el = df_asp_weekly[
    (df_asp_weekly['sub_brand'] == ARIEL_GEL) &
    (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))
].copy()
ariel_weekly_el['trial_month'] = (
    ariel_weekly_el['week_end'].dt.to_period('M').dt.to_timestamp()
)
_el_ariel = df_elasticity[df_elasticity['sub_brand'] == ARIEL_GEL][
    ['trial_month', 'size_code', 'trial_rate', 'subbrand_trial_shoppers']
].drop_duplicates()
ariel_weekly_el = ariel_weekly_el.merge(
    _el_ariel, on=['trial_month', 'size_code'], how='inner'
)

# ── Weekly data for Chart B scatter ───────────────────────────────────
ariel_weekly = df_asp_weekly[
    (df_asp_weekly['sub_brand'] == ARIEL_GEL) &
    (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))
].copy()

ariel_m_sizes = order_and_filter_sizes(
    [s for s in ariel_weekly_el['size_code'].unique()
     if len(ariel_weekly_el[ariel_weekly_el['size_code'] == s]) >= 10]
)
ariel_w_sizes = order_and_filter_sizes(
    [s for s in ariel_weekly['size_code'].unique()
     if len(ariel_weekly[ariel_weekly['size_code'] == s]) >= 10]
)

MIN_BIN_OBS = 2    # min weekly obs per 50 JPY bin — keeps edge price points
                   # ⚠️ INTENTIONAL EXCEPTION: lower than NB03/NB04 because weekly×retailer
                   #    granularity provides denser coverage per bin (see NB04: MIN_FREQ=30)

# ── Shared color palette (cross-notebook standard) ────────────────────
PROD_COL_HI = '#2980B9'   # sky blue  – above-avg productivity (semantic: good)
PROD_COL_LO = '#E67E22'   # amber     – below-avg productivity (semantic: caution)
PROD_COL_OK = '#95A5A6'   # gray      – near-average

def fmt_km(v):
    if v >= 1_000_000: return f'{v/1_000_000:.1f}M'
    if v >= 1_000:     return f'{v/1_000:.1f}K'
    return str(int(v))

# ════════════════════════════════════════════════════════════════════════
# CHART A — Price Point Productivity Index (full price range)
# ════════════════════════════════════════════════════════════════════════
print('=' * 80)
print('CHART A: Price Point Productivity — Trial Rate % per 50 JPY ASP Bin')
print('  Metric : avg trial rate when ASP falls in that bin (normalised by category traffic)')
print('  Index  : bin rate ÷ overall avg × 100  |  Opacity = observation density')
print('  Blue   : above avg productivity  |  Amber : below avg  |  Gray : near avg')
print('=' * 80)

n_s = len(ariel_m_sizes)
n_c = min(2, n_s)
n_r = (n_s + n_c - 1) // n_c

fig_a = make_subplots(
    rows=n_r, cols=n_c,
    subplot_titles=[f'{s}' for s in ariel_m_sizes],
    vertical_spacing=0.18, horizontal_spacing=0.14,
    specs=[[{'secondary_y': True}] * n_c for _ in range(n_r)]
)

strategic_prices = {}

def bin_color(idx_val):
    if idx_val >= 105: return PROD_COL_HI
    if idx_val <= 95:  return PROD_COL_LO
    return PROD_COL_OK

for idx, size in enumerate(ariel_m_sizes):
    row = idx // n_c + 1
    col = idx % n_c + 1
    s = ariel_weekly_el[ariel_weekly_el['size_code'] == size].copy()

    s['asp_bin'] = (s['weighted_asp'] // 50 * 50).astype(int)
    bin_agg = s.groupby('asp_bin').agg(
        avg_trial_rate=('trial_rate', 'mean'),
        avg_buyers=('subbrand_trial_shoppers', 'mean'),
        frequency=('trial_rate', 'count')
    ).reset_index()
    bin_agg = bin_agg[bin_agg['frequency'] >= MIN_BIN_OBS].sort_values('asp_bin')

    if len(bin_agg) == 0:
        continue

    overall_avg_rate = bin_agg['avg_trial_rate'].mean()
    bin_agg['prod_index'] = (bin_agg['avg_trial_rate'] / overall_avg_rate * 100).round(1)
    bar_colors = [bin_color(v) for v in bin_agg['prod_index']]

    # Opacity scaled by observation density (sparse bins appear lighter)
    max_freq = bin_agg['frequency'].max()
    bar_opacities = [max(0.35, min(1.0, f / max_freq * 1.5)) for f in bin_agg['frequency']]

    peak_row = bin_agg.loc[bin_agg['avg_trial_rate'].idxmax()]
    strategic_prices[size] = (int(peak_row['asp_bin']), round(peak_row['avg_trial_rate'], 2))

    x_labels = [f'¥{v}~' for v in bin_agg['asp_bin']]

    # Bar: Productivity Index (left Y)
    fig_a.add_trace(go.Bar(
        x=x_labels,
        y=bin_agg['prod_index'],
        text=[f'{v:.0f}' for v in bin_agg['prod_index']],
        textposition='outside',
        textfont=dict(size=11),
        marker=dict(color=bar_colors, opacity=bar_opacities),
        name=f'{size} Productivity Index',
        showlegend=False,
        customdata=np.stack([bin_agg['avg_trial_rate'],
                             bin_agg['frequency'],
                             bin_agg['avg_buyers']], axis=1),
        hovertemplate=(
            '%{x}<br>'
            'Productivity Index: %{y:.0f}<br>'
            'Trial Rate: %{customdata[0]:.2f}%<br>'
            'Weekly obs (opacity=density): %{customdata[1]:.0f}<br>'
            'Avg Buyers: %{customdata[2]:.0f}<extra></extra>'
        )
    ), secondary_y=False, row=row, col=col)

    # Line: Trial Rate % (right Y — absolute reference)
    fig_a.add_trace(go.Scatter(
        x=x_labels,
        y=bin_agg['avg_trial_rate'],
        mode='lines+markers',
        line=dict(color='#2C3E50', width=1.5, dash='dot'),
        marker=dict(size=7, color='#2C3E50'),
        name=f'{size} Trial Rate %',
        showlegend=False,
        hovertemplate='Trial Rate: %{y:.2f}%<extra></extra>'
    ), secondary_y=True, row=row, col=col)

    # Parity at 100
    fig_a.add_hline(y=100, line_dash='dash', line_color='#BDC3C7', line_width=1.2,
                    row=row, col=col, secondary_y=False)

    # Annotate peak
    fig_a.add_annotation(
        x=f'¥{int(peak_row["asp_bin"])}~',
        y=peak_row['prod_index'],
        text=f'★ Best<br>Index {peak_row["prod_index"]:.0f}',
        showarrow=True, arrowhead=2, arrowcolor=PROD_COL_HI,
        font=dict(color=PROD_COL_HI, size=10, family='Arial Black'),
        ax=0, ay=-40,
        row=row, col=col
    )

    fig_a.update_xaxes(title_text='ASP (50 JPY bin)', tickfont=dict(size=11), row=row, col=col)
    fig_a.update_yaxes(title_text='Productivity Index', secondary_y=False,
                       tickfont=dict(size=11), row=row, col=col)
    fig_a.update_yaxes(title_text='Trial Rate %', secondary_y=True,
                       showgrid=False, tickfont=dict(size=10), row=row, col=col)

fig_a.update_layout(
    height=420 * n_r,
    title=dict(
        text=(
            'アリエールジェル: Price Point Productivity by Size<br>'
            '<sup>'
            '<span style="color:#2980B9">■ Index &gt;105 → above-avg conversion</span>  '
            '<span style="color:#E67E22">■ Index &lt;95 → below-avg conversion</span>  '
            '★ = most productive price  '
            '--- = parity 100  '
            'opacity = observation density'
            '</sup>'
        ),
        font=dict(size=15)
    ),
    template='plotly_white',
    font=dict(size=13)
)
fig_a.show()

# ── Productivity summary table ─────────────────────────────────────────
print('\n💡 Most Productive Price Point per Size (Ariel Gel):')
print(f'  {"Size":30s}  {"Best Price Bin":16s}  {"Trial Rate %":>12s}  {"Productivity"}')
print('  ' + '-' * 75)
for size in ariel_m_sizes:
    s = ariel_weekly_el[ariel_weekly_el['size_code'] == size].copy()
    s['asp_bin'] = (s['weighted_asp'] // 50 * 50).astype(int)
    bg = s.groupby('asp_bin').agg(avg_tr=('trial_rate', 'mean'), n=('trial_rate', 'count')).reset_index()
    bg = bg[bg['n'] >= MIN_BIN_OBS]
    if len(bg) == 0:
        continue
    overall_avg_r = bg['avg_tr'].mean()
    peak = bg.loc[bg['avg_tr'].idxmax()]
    idx_val = peak['avg_tr'] / overall_avg_r * 100
    print(f'  {size:30s}  ¥{int(peak["asp_bin"]):>5,}~{int(peak["asp_bin"])+49:<5,}  '
          f'{peak["avg_tr"]:>10.2f}%  Index {idx_val:.0f}')

# ════════════════════════════════════════════════════════════════════════
# CHART B — Weekly ASP vs Buyers scatter (raw execution view)
# ════════════════════════════════════════════════════════════════════════
print('\n' + '=' * 80)
print('CHART B: ASP vs Buyer Count Scatter (weekly×retailer, 5th–95th pct clip)')
print('Note: count-based, not normalised — use Chart A for productivity.')
print('=' * 80)

fig_b = make_subplots(
    rows=n_r, cols=n_c,
    subplot_titles=[f'{s}: ASP vs Buyers (r = correlation)' for s in ariel_w_sizes],
    vertical_spacing=0.15, horizontal_spacing=0.12
)

for idx, size in enumerate(ariel_w_sizes):
    row = idx // n_c + 1
    col = idx % n_c + 1
    subset = ariel_weekly[ariel_weekly['size_code'] == size].copy()
    if len(subset) < 10:
        continue

    x_lo, x_hi = subset['weighted_asp'].quantile([0.05, 0.95])
    y_lo, y_hi = subset['buyer_count'].quantile([0.05, 0.95])
    sc = subset[subset['weighted_asp'].between(x_lo, x_hi) & subset['buyer_count'].between(y_lo, y_hi)]
    corr = sc['weighted_asp'].corr(sc['buyer_count'])

    scatter_col = PROD_COL_HI if corr < -0.1 else PROD_COL_LO
    fig_b.add_trace(go.Scatter(
        x=sc['weighted_asp'], y=sc['buyer_count'],
        mode='markers', name=f'{size} (r={corr:.2f})',
        marker=dict(size=5, opacity=0.4, color=scatter_col),
        showlegend=True
    ), row=row, col=col)

    if len(sc) >= 3:
        z = np.polyfit(sc['weighted_asp'], sc['buyer_count'], 1)
        p_fn = np.poly1d(z)
        x_line = np.linspace(sc['weighted_asp'].min(), sc['weighted_asp'].max(), 50)
        direction = '↓ lower price → more buyers' if z[0] < 0 else '↑ price not sole driver'
        fig_b.add_trace(go.Scatter(
            x=x_line, y=p_fn(x_line), mode='lines',
            line=dict(dash='dash', color='#E74C3C', width=2),
            name=direction, showlegend=(idx == 0)
        ), row=row, col=col)

    fig_b.update_xaxes(range=[x_lo * 0.99, x_hi * 1.01],
                       title_text='ASP (JPY)', tickfont=dict(size=11), row=row, col=col)
    fig_b.update_yaxes(range=[max(0, y_lo * 0.95), y_hi * 1.05],
                       title_text='Buyers', tickfont=dict(size=11), row=row, col=col)

fig_b.update_layout(
    height=370 * n_r,
    title=dict(
        text='アリエールジェル: Weekly×Retailer ASP vs Buyers — Execution View (5th–95th pct)',
        font=dict(size=15)
    ),
    template='plotly_white',
    font=dict(size=13)
)
fig_b.show()

print('\nASP–Buyer Correlation by Size:')
for size in ariel_w_sizes:
    s2 = ariel_weekly[ariel_weekly['size_code'] == size]
    if len(s2) < 10:
        continue
    xl, xh = s2['weighted_asp'].quantile([0.05, 0.95])
    yl, yh = s2['buyer_count'].quantile([0.05, 0.95])
    sc2 = s2[s2['weighted_asp'].between(xl, xh) & s2['buyer_count'].between(yl, yh)]
    r = sc2['weighted_asp'].corr(sc2['buyer_count'])
    sp = strategic_prices.get(size, ('?', '?'))
    print(f'  {size:30s}  r={r:+.3f}  {"↓ lower→more buyers" if r < -0.1 else "→ price not sole driver":24s}  '
          f'Most productive bin: ¥{sp[0]}~{sp[0]+49 if isinstance(sp[0], int) else "?"}  '
          f'(trial rate {sp[1]:.2f}%)')


CHART A: Price Point Productivity — Trial Rate % per 50 JPY ASP Bin
  Metric : avg trial rate when ASP falls in that bin (normalised by category traffic)
  Index  : bin rate ÷ overall avg × 100  |  Opacity = observation density
  Blue   : above avg productivity  |  Amber : below avg  |  Gray : near avg



💡 Most Productive Price Point per Size (Ariel Gel):
  Size                            Best Price Bin    Trial Rate %  Productivity
  ---------------------------------------------------------------------------
  本体通常                            ¥  150~199         44.88%  Index 109
  詰替超特大                           ¥  250~299         26.94%  Index 103
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                    ¥  350~399         28.82%  Index 131
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   ¥  650~699         21.37%  Index 104
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                     ¥  750~799         27.76%  Index 126

CHART B: ASP vs Buyer Count Scatter (weekly×retailer, 5th–95th pct clip)
Note: count-based, not normalised — use Chart A for productivity.



ASP–Buyer Correlation by Size:
  本体通常                            r=-0.554  ↓ lower→more buyers       Most productive bin: ¥150~199  (trial rate 44.88%)
  詰替超特大                           r=-0.383  ↓ lower→more buyers       Most productive bin: ¥250~299  (trial rate 26.94%)
  詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                    r=-0.172  ↓ lower→more buyers       Most productive bin: ¥350~399  (trial rate 28.82%)
  詰替超ｳﾙﾄﾗｼﾞｬﾝﾎﾞ                   r=+0.107  → price not sole driver   Most productive bin: ¥650~699  (trial rate 21.37%)
  詰替ﾒｶﾞｼﾞｬﾝﾎﾞ                     r=-0.211  ↓ lower→more buyers       Most productive bin: ¥750~799  (trial rate 27.76%)


## 5. Price Gap Heatmaps (50 JPY bins)
Each heatmap pairs **Ariel** and **Attack 抗菌EX** ASPs by week × retailer, floored to 50 JPY bands.
- **Heatmap A** — Value = Ariel buyer count (proxy for trial acquisition velocity)
- **Heatmap B** — Value = Ariel unit index vs Attack (Ariel units ÷ Attack units × 100)

In [28]:

# ── [DS] Cell 20: Price Gap Heatmaps (50 JPY bins) ────────────────────
# Y-axis: Attack ASP high at TOP, low at BOTTOM  (↑ = higher price, intuitive)
# Frequency filter: MIN_OBS_FREQ = store×week count per cell
#   cells below threshold → shown as null (blank) to avoid misleading data

import plotly.express as px

MIN_OBS_FREQ = 10   # minimum store×week observations per (ariel_bin, attack_bin) cell
                    # raised from 5 → 10 to reduce noise from infrequent price combos

# Pair Ariel and Attack at weekly × retailer × size level
ariel_w = df_asp_weekly[
    (df_asp_weekly['sub_brand'] == ARIEL_GEL) &
    (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))
][['week_end', 'retailer', 'size_code', 'weighted_asp', 'total_units', 'buyer_count']].copy()

attack_w = df_asp_weekly[
    (df_asp_weekly['sub_brand'] == ATTACK_EX) &
    (~df_asp_weekly['size_code'].isin(EXCLUDED_SIZES))
][['week_end', 'retailer', 'size_code', 'weighted_asp', 'total_units']].copy()

paired = ariel_w.merge(
    attack_w, on=['week_end', 'retailer', 'size_code'],
    suffixes=('_ariel', '_attack'), how='inner'
)
print(f'Paired observations (week × retailer × size): {len(paired):,}')

# Floor ASPs to 50 JPY bins
paired['ariel_asp_bin'] = (paired['weighted_asp_ariel'] // 50 * 50).astype(int)
paired['attack_asp_bin'] = (paired['weighted_asp_attack'] // 50 * 50).astype(int)
paired['unit_index']    = paired['total_units_ariel'] / paired['total_units_attack'] * 100

# Ordered + filtered sizes
sizes_for_hm = order_and_filter_sizes(paired['size_code'].unique())


def _apply_freq_filter(df_size, group_cols, value_col, aggfunc, fill):
    """Pivot with frequency mask: cells < MIN_OBS_FREQ → NaN."""
    obs = df_size.groupby(group_cols).size().reset_index(name='_obs')
    d   = df_size.merge(obs, on=group_cols, how='left')
    d.loc[d['_obs'] < MIN_OBS_FREQ, value_col] = np.nan
    hm = d.pivot_table(
        index=group_cols[0], columns=group_cols[1],
        values=value_col, aggfunc=aggfunc, fill_value=fill
    )
    # Replace 0 fill_value that came from explicit zeros with nan for masked rows
    if fill == np.nan:
        pass  # already nan
    # Sort Y-axis DESCENDING → higher Attack ASP at TOP (intuitive price axis)
    hm = hm.sort_index(ascending=False)
    return hm, obs


# ═══ Heatmap A: Ariel Buyer Count ════════════════════════════════════
print('\n' + '=' * 80)
print(f'Heatmap A: Price Gap × Ariel Buyer Count  (MIN_OBS_FREQ={MIN_OBS_FREQ} store×week)')
print('Y-axis: Attack ASP (high ↑ → low ↓)  |  Blank cells = below frequency threshold')
print('=' * 80)

for size in sizes_for_hm:
    s = paired[paired['size_code'] == size].copy()
    if len(s) < MIN_OBS_FREQ:
        print(f'  Skipping {size}: too few total observations ({len(s)})')
        continue

    hm, obs_df = _apply_freq_filter(
        s, ['attack_asp_bin', 'ariel_asp_bin'], 'buyer_count', 'sum', np.nan
    )
    # Restore NaN (fill_value=np.nan doesn't work—use explicit mask post-pivot)
    obs_pivot = obs_df.pivot_table(
        index='attack_asp_bin', columns='ariel_asp_bin',
        values='_obs', fill_value=0
    ).reindex(index=hm.index, columns=hm.columns, fill_value=0)
    hm_masked = hm.where(obs_pivot >= MIN_OBS_FREQ, other=np.nan)
    hm_masked = hm_masked.sort_index(ascending=False)

    if hm_masked.isnull().all().all():
        print(f'  Skipping {size}: all cells below frequency threshold')
        continue

    # Text annotation: show buyer count + obs in parentheses
    text_arr = []
    for att_bin in hm_masked.index:
        row_txt = []
        for ar_bin in hm_masked.columns:
            v = hm_masked.loc[att_bin, ar_bin]
            o = obs_pivot.loc[att_bin, ar_bin] if (att_bin in obs_pivot.index and ar_bin in obs_pivot.columns) else 0
            if pd.isna(v) or o < MIN_OBS_FREQ:
                row_txt.append('')
            else:
                row_txt.append(f'{int(v):,}<br>({int(o)}obs)')
        text_arr.append(row_txt)

    fig = px.imshow(
        hm_masked, text_auto=False, aspect='auto',
        labels=dict(x='アリエール ASP (50JPY bin)', y='アタック抗菌EX ASP (50JPY bin)',
                    color='Ariel Buyers'),
        title=f'【{size}】Price Gap × Ariel Buyer Count  (Y↑=high Attack ASP)',
        color_continuous_scale='YlOrRd'
    )
    # Overlay custom text
    fig.update_traces(text=text_arr, texttemplate='%{text}')

    # Y-axis: already sorted descending; ensure axis direction is normal (not reversed by imshow)
    fig.update_yaxes(autorange=True)
    fig.update_layout(width=800, height=580)
    fig.show()

# ═══ Heatmap B: Ariel Unit Index vs Attack ═══════════════════════════
print('\n' + '=' * 80)
print(f'Heatmap B: Price Gap × Ariel Unit Index vs Attack  (MIN_OBS_FREQ={MIN_OBS_FREQ})')
print('Green=Ariel wins | Red=Attack wins | Blank=low frequency (unreliable)')
print('=' * 80)

for size in sizes_for_hm:
    s = paired[paired['size_code'] == size].copy()
    if len(s) < MIN_OBS_FREQ:
        print(f'  Skipping {size}: too few observations ({len(s)})')
        continue

    obs_counts = s.groupby(['attack_asp_bin', 'ariel_asp_bin']).size().reset_index(name='obs_count')
    s = s.merge(obs_counts, on=['attack_asp_bin', 'ariel_asp_bin'], how='left')
    # Mask low-frequency rows before pivoting
    s.loc[s['obs_count'] < MIN_OBS_FREQ, 'unit_index'] = np.nan

    hm = s.pivot_table(
        index='attack_asp_bin', columns='ariel_asp_bin',
        values='unit_index', aggfunc='mean'
    )
    # Y-axis DESCENDING → higher Attack ASP at TOP
    hm = hm.sort_index(ascending=False)

    obs_pivot = obs_counts.pivot_table(
        index='attack_asp_bin', columns='ariel_asp_bin',
        values='obs_count', fill_value=0
    ).reindex(index=hm.index, columns=hm.columns, fill_value=0)

    if hm.isnull().all().all():
        print(f'  Skipping {size}: all cells below frequency threshold')
        continue

    # Text: unit index + obs count
    text_arr = []
    for att_bin in hm.index:
        row_txt = []
        for ar_bin in hm.columns:
            v = hm.loc[att_bin, ar_bin] if (att_bin in hm.index and ar_bin in hm.columns) else np.nan
            o = obs_pivot.loc[att_bin, ar_bin] if (att_bin in obs_pivot.index and ar_bin in obs_pivot.columns) else 0
            if pd.isna(v) or o < MIN_OBS_FREQ:
                row_txt.append('')
            else:
                row_txt.append(f'{v:.0f}<br>({int(o)}obs)')
        text_arr.append(row_txt)

    fig = px.imshow(
        hm, text_auto=False, aspect='auto',
        labels=dict(x='アリエール ASP (50JPY bin)', y='アタック抗菌EX ASP (50JPY bin)',
                    color='Unit Index (Ariel/Attack×100)'),
        title=(f'【{size}】Price Gap × Ariel Unit Index vs Attack  '
               f'(Y↑=high Attack ASP | ≥{MIN_OBS_FREQ} obs required)'),
        color_continuous_scale='RdYlGn',
        zmin=50, zmax=150   # centre at 100 = parity
    )
    fig.update_traces(text=text_arr, texttemplate='%{text}')
    fig.update_yaxes(autorange=True)
    fig.update_layout(width=800, height=580)
    fig.show()

# ── Summary table: optimal price combinations ─────────────────────────
print('\n' + '=' * 80)
print(f'Optimal Price Combinations (≥{MIN_OBS_FREQ} obs, sorted by Ariel buyer count)')
print('=' * 80)
for size in sizes_for_hm:
    s = paired[paired['size_code'] == size]
    if len(s) < MIN_OBS_FREQ:
        continue
    top = s.groupby(['ariel_asp_bin', 'attack_asp_bin']).agg(
        total_buyers=('buyer_count', 'sum'),
        mean_unit_index=('unit_index', 'mean'),
        obs_count=('week_end', 'count')
    ).reset_index()
    top = top[top['obs_count'] >= MIN_OBS_FREQ].sort_values('total_buyers', ascending=False).head(5)

    print(f'\n▶ {size}   (Y-axis direction: higher Attack ASP shown at top in heatmap):')
    print(f'  {"Ariel ASP":>12s} {"Attack ASP":>12s} {"Buyers":>10s} {"UnitIdx":>9s} {"Obs(storeXwk)":>14s}')
    for _, rw in top.iterrows():
        print(f'  ¥{int(rw["ariel_asp_bin"]):>5,}~{int(rw["ariel_asp_bin"])+49:<4}  '
              f'¥{int(rw["attack_asp_bin"]):>5,}~{int(rw["attack_asp_bin"])+49:<4}  '
              f'{int(rw["total_buyers"]):>10,} {rw["mean_unit_index"]:>9.0f} {int(rw["obs_count"]):>14,}')


Paired observations (week × retailer × size): 13,026

Heatmap A: Price Gap × Ariel Buyer Count  (MIN_OBS_FREQ=10 store×week)
Y-axis: Attack ASP (high ↑ → low ↓)  |  Blank cells = below frequency threshold



Heatmap B: Price Gap × Ariel Unit Index vs Attack  (MIN_OBS_FREQ=10)
Green=Ariel wins | Red=Attack wins | Blank=low frequency (unreliable)



Optimal Price Combinations (≥10 obs, sorted by Ariel buyer count)

▶ 本体通常   (Y-axis direction: higher Attack ASP shown at top in heatmap):
     Ariel ASP   Attack ASP     Buyers   UnitIdx  Obs(storeXwk)
  ¥  200~249   ¥  350~399      360,625      3772            108
  ¥  150~199   ¥  200~249      170,756       575             73
  ¥  150~199   ¥  300~349      164,417       933             70
  ¥  200~249   ¥  300~349      101,173       298            128
  ¥  250~299   ¥  200~249      100,840       105            118

▶ 詰替超特大   (Y-axis direction: higher Attack ASP shown at top in heatmap):
     Ariel ASP   Attack ASP     Buyers   UnitIdx  Obs(storeXwk)
  ¥  300~349   ¥  350~399      982,256      1526            582
  ¥  300~349   ¥  300~349      626,652       248            467
  ¥  350~399   ¥  350~399      616,599       118            600
  ¥  250~299   ¥  350~399      479,239       229            123
  ¥  300~349   ¥  400~449      340,976      2204            225

▶ 詰替ｳﾙﾄﾗｼﾞｬﾝﾎﾞ   

In [15]:
# ── Export Phase 2 results ────────────────────────────────────────────
output_file = 'phase2_trial_acquisition.xlsx'

# Strip timezone for Excel compatibility
def strip_tz(df):
    df = df.copy()
    for col in df.select_dtypes(include=['datetimetz']).columns:
        df[col] = df[col].dt.tz_localize(None)
    # Also convert Period / Interval types to string
    for col in df.columns:
        if df[col].dtype == 'object':
            try:
                df[col] = df[col].astype(str)
            except:
                pass
    return df

with pd.ExcelWriter(output_file, engine='openpyxl') as writer:
    strip_tz(df_trial).to_excel(writer, sheet_name='Trial_Monthly', index=False)
    strip_tz(summary).to_excel(writer, sheet_name='Trial_Summary', index=False)
    strip_tz(df_elasticity).to_excel(writer, sheet_name='ASP_Elasticity', index=False)
    strip_tz(df_asp_weekly).to_excel(writer, sheet_name='Weekly_ASP_Retailer', index=False)
    strip_tz(paired).to_excel(writer, sheet_name='Price_Gap_Paired', index=False)

print(f'✅ Phase 2 results exported to {output_file}')
print(f'   Sheets: Trial_Monthly, Trial_Summary, ASP_Elasticity, Weekly_ASP_Retailer, Price_Gap_Paired')

✅ Phase 2 results exported to phase2_trial_acquisition.xlsx
   Sheets: Trial_Monthly, Trial_Summary, ASP_Elasticity, Weekly_ASP_Retailer, Price_Gap_Paired
